<a href="https://colab.research.google.com/github/Viniciusp2/predicao-radiacao-solar-para/blob/main/notbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Cell 0 - Dependências
!pip install -q unidecode pyarrow

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 235.8/235.8 kB 2.8 MB/s eta 0:00:00


In [ ]:
# 1) ETL multi-arquivos — lê, limpa, consolida (cidade robusta)
import os, re, glob
from typing import Dict, Tuple, List
import numpy as np
import pandas as pd
from unidecode import unidecode

INPUT_DIR = "/content/drive/MyDrive/inmet_csvs"
OUTPUT_DIR = "./saidas_inmet"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "limpos_csv"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "limpos_parquet"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "relatorios"), exist_ok=True)

UF_SET = {"AC","AL","AP","AM","BA","CE","DF","ES","GO","MA","MT","MS","MG","PA","PB","PR","PE","PI","RJ","RN","RS","RO","RR","SC","SP","SE","TO"}
REG_SET = {"N","NE","CO","SE","S","SU","SD","O","L"}
STOP_TOKENS = {"INMET","DADOS","HORARIA","HORARIO","SERIE","ESTACAO","AUT","MET","CSV"}
WMO_PAT = re.compile(r"^[A-Z]?\d{3,5}$", re.IGNORECASE)
SHORT_PAT = re.compile(r"^[A-Za-z]{1,2}$")

def normalize_colname(name: str) -> str:
    if name is None: return ""
    s = unidecode(str(name)).lower().replace("%","pct")
    s = re.sub(r"[^a-z0-9]+","_",s)
    return re.sub(r"_+","_",s).strip("_")

def standardize_known_columns(cols: List[str]) -> List[str]:
    mapping = {
        "data":"data","data_yyyy_mm_dd":"data",
        "hora":"hora","hora_utc":"hora",
        "precipitacao_total":"precipitacao_total_mm","precipitacao_total_horario_mm":"precipitacao_total_mm","precipitacao_total_mm":"precipitacao_total_mm",
        "pressao_atmosferica":"pressao_atm_mb","pressao_atmosferica_ao_nivel_da_estacao_horaria_mb":"pressao_atm_mb",
        "pressao_atmosferica_max_na_hora_ant_aut_mb":"pressao_atmosferica_max_na_hora_ant_aut_mb",
        "pressao_atmosferica_min_na_hora_ant_aut_mb":"pressao_atmosferica_min_na_hora_ant_aut_mb",
        "radiacao_global":"radiacao_global_kj_m2","radiacao_global_kj_m2":"radiacao_global_kj_m2",
        "temperatura_do_ar_bulbo_seco_horaria_c":"temperatura_c","temperatura_do_ar":"temperatura_c",
        "temperatura_do_ponto_de_orvalho_c":"temperatura_do_ponto_de_orvalho_degc",
        "umidade":"umidade_rel_pct","umidade_relativa_do_ar_horaria_pct":"umidade_rel_pct",
        "vento_velocidade_m_s":"vento_vel_m_s","vento_velocidade":"vento_vel_m_s","vento":"vento_vel_m_s",
        "vento_direcao":"vento_dir_graus","vento_direcao_horaria_gr_deg_gr":"vento_direcao_horaria_gr_deg_gr",
        "vento_rajada_maxima_m_s":"vento_rajada_maxima_m_s","vento_velocidade_horaria_m_s":"vento_velocidade_horaria_m_s",
        "temperatura_maxima_na_hora_ant_aut_c":"temperatura_maxima_na_hora_ant_aut_degc",
        "temperatura_minima_na_hora_ant_aut_c":"temperatura_minima_na_hora_ant_aut_degc",
        "temperatura_orvalho_max_na_hora_ant_aut_c":"temperatura_orvalho_max_na_hora_ant_aut_degc",
        "temperatura_orvalho_min_na_hora_ant_aut_c":"temperatura_orvalho_min_na_hora_ant_aut_degc",
        "umidade_rel_max_na_hora_ant_aut_pct":"umidade_rel_max_na_hora_ant_aut_pct",
        "umidade_rel_min_na_hora_ant_aut_pct":"umidade_rel_min_na_hora_ant_aut_pct",
    }
    return [mapping.get(normalize_colname(c), normalize_colname(c)) for c in cols]

def read_metadata(path: str, nlines: int = 8) -> Dict[str, str]:
    meta = {}
    with open(path,"r",encoding="latin1",errors="ignore") as f:
        for i in range(nlines):
            line = f.readline()
            if not line: break
            parts = re.split(r"[:;]", line, maxsplit=1)
            if len(parts)==2: key,val = parts
            else:
                parts = re.split(r",", line, maxsplit=1)
                key,val = (parts if len(parts)==2 else (f"linha_{i+1}", line))
            meta[normalize_colname(key)] = val.strip().strip(";:, ").replace("\n","")
    meta["arquivo"] = os.path.basename(path)
    return meta

def parse_hour_to_hhmm(hour_raw: str) -> str:
    if pd.isna(hour_raw): return None
    s = str(hour_raw).strip().lower().replace("utc","").strip()
    s = re.sub(r"[^0-9:]","",s)
    if s=="": return None
    if re.fullmatch(r"\d{4}", s): return f"{s[:2]}:{s[2:]}"
    if re.fullmatch(r"\d{1,2}", s): return f"{int(s):02d}:00"
    if re.fullmatch(r"\d{1,2}:\d{2}", s):
        hh,mm = map(int, s.split(":"))
        if 0<=hh<=23 and 0<=mm<=59: return f"{hh:02d}:{mm:02d}"
    return None

def build_datetime_utc(df: pd.DataFrame, col_date="data", col_hour="hora") -> pd.Series:
    date_str = pd.to_datetime(df[col_date], errors="coerce").dt.strftime("%Y-%m-%d")
    hour_norm = df[col_hour].apply(parse_hour_to_hhmm)
    return pd.to_datetime((date_str.fillna("")+" "+hour_norm.fillna("")).str.strip(),
                          format="%Y-%m-%d %H:%M", errors="coerce")

def coerce_numeric(df: pd.DataFrame, exclude: List[str]) -> pd.DataFrame:
    for c in df.columns:
        if c in exclude:
            continue
        if df[c].dtype == object:
            df[c] = pd.to_numeric(df[c].astype(str).str.replace(",",".", regex=False), errors="coerce")
        else:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df

def treat_missing_and_outliers(df: pd.DataFrame) -> pd.DataFrame:
    for c in df.columns:
        if df[c].dtype == object:
            df[c] = df[c].replace({-9999: np.nan, "////": np.nan, "": np.nan})
    if "umidade_rel_pct" in df.columns:
        df.loc[(df["umidade_rel_pct"]<0)|(df["umidade_rel_pct"]>100),"umidade_rel_pct"]=np.nan
    if "temperatura_c" in df.columns:
        df.loc[(df["temperatura_c"]<-50)|(df["temperatura_c"]>60),"temperatura_c"]=np.nan
    if "radiacao_global_kj_m2" in df.columns:
        df.loc[df["radiacao_global_kj_m2"]<0,"radiacao_global_kj_m2"]=np.nan
    if "vento_vel_m_s" in df.columns:
        df.loc[(df["vento_vel_m_s"]<0)|(df["vento_vel_m_s"]>60),"vento_vel_m_s"]=np.nan
    if "precipitacao_total_mm" in df.columns:
        df.loc[df["precipitacao_total_mm"]<0,"precipitacao_total_mm"]=np.nan
    if "pressao_atm_mb" in df.columns:
        df.loc[(df["pressao_atm_mb"]<800)|(df["pressao_atm_mb"]>1100),"pressao_atm_mb"]=np.nan
    return df

def missing_report(df: pd.DataFrame) -> pd.DataFrame:
    return (df.isna().mean().sort_values(ascending=False)*100.0).to_frame("pct_missing")

def describe_numeric(df: pd.DataFrame) -> pd.DataFrame:
    return df.select_dtypes(include=[np.number]).describe().T

def detect_header_row(path: str, max_scan: int = 30) -> int:
    lines = []
    with open(path, "r", encoding="latin1", errors="ignore") as f:
        for i in range(max_scan):
            line = f.readline()
            if not line: break
            lines.append(line.strip().lower())
    for i,line in enumerate(lines):
        if ";" in line and ("data" in line) and ("hora" in line):
            return i
    return 8

def clean_meta_number(s: str):
    if s is None: return np.nan
    t = str(s).strip()
    t = re.sub(r"^;","", t)
    t = t.replace(",", ".")
    try:
        return float(t)
    except:
        return np.nan

def _norm_token(t):
    t = unidecode(str(t)).strip()
    t = re.sub(r"[^A-Za-z0-9\- ]"," ",t)
    t = re.sub(r"\s+"," ",t).strip()
    return t

def _is_wmo_like(tok):
    return bool(WMO_PAT.match((tok or "").strip()))

def _plausible_city(tok):
    if not tok: return False
    T = _norm_token(tok).upper()
    if T in STOP_TOKENS: return False
    if T in UF_SET: return False
    if T in REG_SET: return False
    if SHORT_PAT.match(T): return False
    if _is_wmo_like(T): return False
    if any(ch.isdigit() for ch in T): return False
    return True

def _clean_city_label(s):
    if s is None: return None
    s = unidecode(str(s))
    if s.startswith("UF-"):
        return s
    s = re.sub(r"^[;,:/\-\s]+","",s)
    s = re.sub(r"[^A-Za-z0-9\-\s]"," ",s)
    s = re.sub(r"\s+"," ",s).strip()
    return s.title() if s else None

def _city_from_filename(fname):
    base = os.path.splitext(os.path.basename(fname))[0]
    toks = re.split(r"[_\-\.\s]+", _norm_token(base))
    cands = [ _clean_city_label(t) for t in toks if _plausible_city(t) ]
    cands = [c for c in cands if c]
    return cands[-1] if cands else None

def _city_from_text(s):
    if not isinstance(s, str): return None
    raw = unidecode(s)
    raw = re.sub(r"\(.*?\)","", raw)
    raw = re.sub(r"\s+\-\s+.*","", raw)
    toks = re.split(r"[_\-\s]+", raw)
    cands = [ _clean_city_label(t) for t in toks if _plausible_city(t) ]
    cands = [c for c in cands if c]
    return cands[-1] if cands else None

def resolve_city(meta: Dict[str,str]) -> str:
    uf = (meta.get("uf") or meta.get("estado") or "").strip().upper()
    for cand in [
        _city_from_filename(meta.get("arquivo","")),
        _city_from_text(meta.get("municipio") or meta.get("cidade")),
        _city_from_text(meta.get("estacao") or meta.get("estacao_nome")),
    ]:
        if cand and _plausible_city(cand):
            return _clean_city_label(cand) or "Desconhecida"
    return f"UF-{uf}" if uf in UF_SET else "Desconhecida"

def process_single_file(path: str) -> Tuple[pd.DataFrame, Dict[str, str], pd.DataFrame, pd.DataFrame]:
    meta = read_metadata(path, nlines=8)
    header_row = detect_header_row(path, max_scan=30)
    df = pd.read_csv(path, sep=";", header=header_row, encoding="latin1", decimal=",", dtype=str,
                     na_values=["////","","NaN","nan"])
    df.columns = standardize_known_columns(list(df.columns))
    if "data" not in df.columns:
        cand = [c for c in df.columns if c.startswith("data")]
        if cand: df = df.rename(columns={cand[0]:"data"})
    if "hora" not in df.columns:
        if "hora_utc" in df.columns: df = df.rename(columns={"hora_utc":"hora"})
        else:
            cand = [c for c in df.columns if c.startswith("hora")]
            if cand: df = df.rename(columns={cand[0]:"hora"})
    if "data" not in df.columns or "hora" not in df.columns:
        raise ValueError("Colunas 'data'/'hora' não encontradas após padronização.")
    df["datetime_utc"] = build_datetime_utc(df, "data", "hora")
    before = len(df)
    df = df[~df["datetime_utc"].isna()].copy()
    df = coerce_numeric(df, ["data","hora","datetime_utc"])
    df = treat_missing_and_outliers(df)
    df["cidade"] = resolve_city(meta)
    df["uf_meta"] = (meta.get("uf") or meta.get("estado") or "").strip().upper()
    df["estacao_meta"] = unidecode(str(meta.get("estacao") or meta.get("estacao_nome") or "")).strip()
    df["codigo_wmo_meta"] = unidecode(str(meta.get("codigo_wmo") or "")).strip()
    df = df.sort_values("datetime_utc").reset_index(drop=True)
    miss_df = missing_report(df)
    stats_df = describe_numeric(df)
    print("="*80)
    print("Arquivo:", os.path.basename(path))
    print("header_row:", header_row)
    print("linhas_validas:", len(df), "/", before)
    for k in ["regiao","uf","estacao","codigo_wmo","latitude","longitude","altitude"]:
        if k in meta: print(f"{k}: {meta[k]}")
    print(miss_df.head().to_string())
    print(stats_df.head().to_string())
    return df, meta, miss_df, stats_df

def save_per_file_outputs(df, miss_df, stats_df, meta, original_path):
    stem = os.path.splitext(os.path.basename(original_path))[0]
    df.to_csv(os.path.join(OUTPUT_DIR,"limpos_csv",f"{stem}_clean.csv"), index=False, encoding="utf-8")
    try: df.to_parquet(os.path.join(OUTPUT_DIR,"limpos_parquet",f"{stem}_clean.parquet"), index=False)
    except: pass
    miss_df.to_csv(os.path.join(OUTPUT_DIR,"relatorios",f"{stem}_missing.csv"))
    stats_df.to_csv(os.path.join(OUTPUT_DIR,"relatorios",f"{stem}_stats.csv"))
    with open(os.path.join(OUTPUT_DIR,"relatorios",f"{stem}_metadata.txt"),"w",encoding="utf-8") as f:
        for k,v in meta.items(): f.write(f"{k}: {v}\n")

def main_etl():
    patterns = ["*.csv","*.CSV","*.Csv","*.cSv","*.csV","*.CSv","*.cSV","*.CsV"]
    files = []
    for pat in patterns:
        files.extend(glob.glob(os.path.join(INPUT_DIR, pat)))
    files = sorted(list(dict.fromkeys(files)))
    if not files:
        print("Nenhum arquivo encontrado em:", os.path.abspath(INPUT_DIR))
        return
    all_frames = []
    for path in files:
        try:
            df, meta, miss_df, stats_df = process_single_file(path)
            save_per_file_outputs(df, miss_df, stats_df, meta, path)
            df["arquivo_origem"] = os.path.basename(path)
            df["latitude_meta"] = clean_meta_number(meta.get("latitude"))
            df["longitude_meta"] = clean_meta_number(meta.get("longitude"))
            all_frames.append(df)
        except Exception as e:
            print("Erro:", os.path.basename(path), "=>", e)
    if not all_frames:
        print("Nada consolidado.")
        return
    df_all = pd.concat(all_frames, ignore_index=True).sort_values("datetime_utc").reset_index(drop=True)
    rename_map = {
        "temperatura_do_ar_bulbo_seco_horaria_degc": "temperatura_c",
        "temperatura_do_ponto_de_orvalho_degc": "ponto_orvalho_c",
        "temperatura_maxima_na_hora_ant_aut_degc": "temp_max_hora_ant_c",
        "temperatura_minima_na_hora_ant_aut_degc": "temp_min_hora_ant_c",
        "temperatura_orvalho_max_na_hora_ant_aut_degc": "orvalho_max_hora_ant_c",
        "temperatura_orvalho_min_na_hora_ant_aut_degc": "orvalho_min_hora_ant_c",
        "vento_velocidade_horaria_m_s": "vento_vel_m_s",
        "vento_rajada_maxima_m_s": "vento_rajada_m_s",
        "vento_direcao_horaria_gr_deg_gr": "vento_dir_graus",
        "pressao_atmosferica_max_na_hora_ant_aut_mb": "pressao_max_hora_ant_mb",
        "pressao_atmosferica_min_na_hora_ant_aut_mb": "pressao_min_hora_ant_mb",
        "radiacao_global_kj_m2": "radiacao_kj_m2",
    }
    df_all = df_all.rename(columns={k:v for k,v in rename_map.items() if k in df_all.columns})
    drop_unnamed = [c for c in df_all.columns if c.lower().startswith("unnamed")]
    if drop_unnamed: df_all = df_all.drop(columns=drop_unnamed)
    na100 = [c for c in df_all.columns if df_all[c].isna().all()]
    if na100: df_all = df_all.drop(columns=na100)
    out_all_csv = os.path.join(OUTPUT_DIR, "consolidado_inmet.csv")
    df_all.to_csv(out_all_csv, index=False, encoding="utf-8")
    try: df_all.to_parquet(os.path.join(OUTPUT_DIR, "consolidado_inmet.parquet"), index=False)
    except: pass
    miss_all = (df_all.isna().mean().sort_values(ascending=False)*100.0).to_frame("pct_missing")
    stats_all = df_all.select_dtypes(include=[np.number]).describe().T
    miss_all.to_csv(os.path.join(OUTPUT_DIR,"relatorios","consolidado_missing.csv"))
    stats_all.to_csv(os.path.join(OUTPUT_DIR,"relatorios","consolidado_stats.csv"))
    print("\nHEAD consolidado:")
    print(df_all.head().to_string())
    print("\nEstatísticas:")
    print(stats_all.to_string())
    print("\nRemovidas colunas 100% NaN:", na100)
    print("Removidas colunas 'unnamed_*':", drop_unnamed)
    print("\nSaídas em:", os.path.abspath(OUTPUT_DIR))

main_etl()


Arquivo: INMET_N_PA_A201_BELEM_01-01-2010_A_31-12-2010.CSV
header_row: 8
linhas_validas: 8760 / 8760
regiao: N
uf: PA
estacao: BELEM
codigo_wmo: A201
latitude: -1,41027777
longitude: -48,43833333
altitude: 24
                       pct_missing
unnamed_19              100.000000
radiacao_global_kj_m2    47.659817
pressao_atm_mb            0.148402
umidade_rel_pct           0.148402
precipitacao_total_mm     0.148402
                                             count         mean          std     min     25%      50%     75%     max
precipitacao_total_mm                       8747.0     0.322602     2.016879     0.0     0.0     0.00     0.0    45.2
pressao_atm_mb                              8747.0  1008.066560     2.022064  1001.2  1006.7  1008.10  1009.5  1015.9
pressao_atmosferica_max_na_hora_ant_aut_mb  8760.0   992.028653   423.750316 -9999.0  1007.0  1008.35  1009.7  1016.0
pressao_atmosferica_min_na_hora_ant_aut_mb  8760.0   991.434395   423.727570 -9999.0  1006.4  1007.70  1009.2

In [ ]:
# Célula 2 — Base diária (06–18h): soma radiação e médias climáticas
import os, pandas as pd, numpy as np

INPUT_DIR = "/content/drive/MyDrive/inmet_csvs"
OUTPUT_DIR = "./saidas_inmet"
TCC_DIR = os.path.join(OUTPUT_DIR, "tcc")
os.makedirs(TCC_DIR, exist_ok=True)

def _load_consolidado():
    cands = [
        os.path.join(OUTPUT_DIR, "consolidado_inmet.parquet"),
        os.path.join(OUTPUT_DIR, "consolidado_inmet.csv"),
    ]
    for p in cands:
        if os.path.exists(p):
            return pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p, low_memory=False)
    return None

def _first_col(df, keys):
    for k in keys:
        if k in df.columns: return k
    for c in df.columns:
        for k in keys:
            if k in c: return c
    return None

df = _load_consolidado()
if df is None or df.empty:
    print("consolidado_inmet não encontrado. Rode a Célula 1 antes.")
else:
    if "datetime_utc" in df.columns:
        dtu = pd.to_datetime(df["datetime_utc"], errors="coerce")
        df["data"] = dtu.dt.date
        df["hora"] = dtu.dt.hour
    else:
        if "data" not in df.columns or "hora" not in df.columns:
            raise ValueError("Sem datetime_utc e sem pares data/hora. Verifique a Célula 1.")
        df["data"] = pd.to_datetime(df["data"], errors="coerce").dt.date
        df["hora"] = pd.to_numeric(df["hora"], errors="coerce")

    df["data"] = pd.to_datetime(df["data"])
    df["hora"] = pd.to_numeric(df["hora"], errors="coerce")

    col_rad = _first_col(df, ["radiacao_kj_m2","radiacao_global_kj_m2","radiacao_global"])
    col_tmp = _first_col(df, ["temperatura_c","temperatura_do_ar"])
    col_umd = _first_col(df, ["umidade_rel_pct","umidade"])
    col_ven = _first_col(df, ["vento_vel_m_s","vento"])
    col_prs = _first_col(df, ["pressao_atm_mb","pressao_atmosferica"])
    col_prc = _first_col(df, ["precipitacao_total_mm","precipitacao_total","precipitacao"])

    if col_rad is None or "cidade" not in df.columns:
        raise ValueError("Colunas essenciais ausentes: 'cidade' e/ou radiação.")

    for c in [col_rad, col_tmp, col_umd, col_ven, col_prs, col_prc]:
        if c is not None:
            if df[c].dtype == object:
                df[c] = pd.to_numeric(df[c].astype(str).str.replace(",",".", regex=False), errors="coerce")
            else:
                df[c] = pd.to_numeric(df[c], errors="coerce")

    base = df[(df["hora"]>=6) & (df["hora"]<=18)].copy()

    agg = {col_rad: "sum"}
    if col_tmp: agg[col_tmp] = "mean"
    if col_umd: agg[col_umd] = "mean"
    if col_ven: agg[col_ven] = "mean"
    if col_prs: agg[col_prs] = "mean"
    if col_prc: agg[col_prc] = "sum"

    daily = (base.groupby(["cidade","data"], as_index=False)
                  .agg(agg))

    rename_map = {}
    if col_rad: rename_map[col_rad] = "radiacao_diaria_kj_m2"
    if col_tmp: rename_map[col_tmp] = "temp_ar_c_media_dia"
    if col_umd: rename_map[col_umd] = "umidade_rel_pct_media_dia"
    if col_ven: rename_map[col_ven] = "vento_m_s_media_dia"
    if col_prs: rename_map[col_prs] = "pressao_mb_media_dia"
    if col_prc: rename_map[col_prc] = "precipitacao_mm_dia"
    daily = daily.rename(columns=rename_map)

    daily["ano"] = daily["data"].dt.year

    daily.to_csv(os.path.join(OUTPUT_DIR, "daily.csv"), index=False, encoding="utf-8")
    daily.to_parquet(os.path.join(OUTPUT_DIR, "daily.parquet"), index=False)
    daily.to_csv(os.path.join(TCC_DIR, "daily.csv"), index=False, encoding="utf-8")
    daily.to_parquet(os.path.join(TCC_DIR, "daily.parquet"), index=False)

    print("daily salvo em:", os.path.join(OUTPUT_DIR, "daily.csv"))
    print("linhas:", len(daily), "| cidades:", daily["cidade"].nunique(), "| anos:", daily["ano"].min(), "→", daily["ano"].max())
    display(daily.head())


In [ ]:
# 3D) Anos presentes por cidade (usando daily)
import os, pandas as pd, numpy as np

OUT_DIR = "./saidas_inmet"; TCC_DIR = os.path.join(OUT_DIR,"tcc")
paths = [os.path.join(TCC_DIR,"daily.parquet"), os.path.join(TCC_DIR,"daily.csv"),
         os.path.join(OUT_DIR,"daily.parquet"), os.path.join(OUT_DIR,"daily.csv")]

daily = None
for p in paths:
    if os.path.exists(p):
        daily = pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p, parse_dates=["data"])
        break
if daily is None or daily.empty: raise RuntimeError("daily não encontrado. Rode a Célula 2.")

if "ano" not in daily.columns:
    daily["ano"] = pd.to_datetime(daily["data"]).dt.year

years_full = list(range(2010, 2020))

pres = (daily.groupby(["cidade","ano"])["data"]
              .nunique().rename("dias").reset_index())
pres["dias_ano"] = pres["ano"].map(lambda a: 366 if a in [2012,2016] else 365)
pres["flag_dia>=250"] = pres["dias"] >= 250

ans = (pres.groupby("cidade")
           .agg(anos_total=("ano","nunique"),
                anos_ok_250dias=("flag_dia>=250","sum"),
                ano_min=("ano","min"),
                ano_max=("ano","max"))
           .reset_index())

anos_list = pres.groupby("cidade")["ano"].apply(lambda s: ",".join(map(str,sorted(set(s))))).rename("anos_presentes")
ans = ans.merge(anos_list, on="cidade", how="left")
ans["anos_faltantes_2010_2019"] = ans["anos_presentes"].apply(
    lambda s: ",".join([str(y) for y in years_full if str(y) not in (s.split(",") if isinstance(s,str) else [])])
)

ans = ans.sort_values(["anos_ok_250dias","anos_total","cidade"], ascending=[False,False,True])
ans.to_csv(os.path.join(TCC_DIR,"auditoria_anos_por_cidade.csv"), index=False, encoding="utf-8")
print("Salvo:", os.path.join(TCC_DIR,"auditoria_anos_por_cidade.csv"))
ans.head(20)


In [ ]:
# Célula 3 — Carrega daily e define parâmetros de cobertura
import os, pandas as pd, numpy as np

OUTPUT_DIR = "./saidas_inmet"
TCC_DIR = os.path.join(OUTPUT_DIR, "tcc")
os.makedirs(TCC_DIR, exist_ok=True)

MIN_DAYS_PER_YEAR = 250
MIN_YEARS_MIN = 7

def _load_daily():
    for p in [os.path.join(TCC_DIR,"daily.parquet"),
              os.path.join(TCC_DIR,"daily.csv"),
              os.path.join(OUTPUT_DIR,"daily.parquet"),
              os.path.join(OUTPUT_DIR,"daily.csv")]:
        if os.path.exists(p):
            return pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p, parse_dates=["data"])
    return None

daily = _load_daily()
if daily is None or daily.empty or "cidade" not in daily.columns or "data" not in daily.columns:
    raise RuntimeError("daily não encontrado/inválido. Execute as Células 1 e 2 antes da 3.")

if "ano" not in daily.columns:
    daily["ano"] = pd.to_datetime(daily["data"]).dt.year

print("daily:", len(daily), "linhas | cidades:", daily["cidade"].nunique(),
      "| período:", daily["data"].min().date(), "→", daily["data"].max().date(),
      "| anos:", daily["ano"].min(), "→", daily["ano"].max())


In [ ]:
# Célula 3B — Filtro por cobertura (gera daily_f + resumo)
import os, pandas as pd, numpy as np

OUTPUT_DIR = "./saidas_inmet"
TCC_DIR = os.path.join(OUTPUT_DIR, "tcc")
os.makedirs(TCC_DIR, exist_ok=True)

try:
    MIN_DAYS_PER_YEAR
    MIN_YEARS_MIN
except NameError:
    MIN_DAYS_PER_YEAR = 250
    MIN_YEARS_MIN = 7

def _load_daily():
    for p in [os.path.join(TCC_DIR,"daily.parquet"),
              os.path.join(TCC_DIR,"daily.csv"),
              os.path.join(OUTPUT_DIR,"daily.parquet"),
              os.path.join(OUTPUT_DIR,"daily.csv")]:
        if os.path.exists(p):
            return pd.read_parquet(p) if p.endswith(".parquet") else pd.read_csv(p, parse_dates=["data"])
    return None

daily = _load_daily()
if daily is None or daily.empty or "cidade" not in daily.columns or "data" not in daily.columns:
    raise RuntimeError("daily não encontrado/inválido. Execute as Células 1 e 2 antes da 3B.")

if "ano" not in daily.columns:
    daily["ano"] = pd.to_datetime(daily["data"]).dt.year

cov = (daily.groupby(["cidade","ano"])["data"]
       .nunique().rename("dias").reset_index())
cov["ok"] = cov["dias"] >= MIN_DAYS_PER_YEAR

anos_ok_por_cidade = cov[cov["ok"]].groupby("cidade")["ano"].nunique()
cidades_passam = anos_ok_por_cidade[anos_ok_por_cidade >= MIN_YEARS_MIN].index.tolist()

if len(cidades_passam) == 0:
    daily_f = daily.copy()
    status = "filtro_desativado_sem_cidades_suficientes"
else:
    daily_f = daily[daily["cidade"].isin(cidades_passam)].copy()
    status = f"filtro_ativado_{len(cidades_passam)}_cidades"

daily_f.to_parquet(os.path.join(TCC_DIR, "daily_f.parquet"), index=False)
daily_f.to_csv(os.path.join(TCC_DIR, "daily_f.csv"), index=False, encoding="utf-8")

resumo = pd.DataFrame({
    "status":[status],
    "min_days_per_year":[MIN_DAYS_PER_YEAR],
    "min_years":[MIN_YEARS_MIN],
    "n_cidades_total":[daily["cidade"].nunique()],
    "n_cidades_filtradas":[daily_f["cidade"].nunique()],
    "ano_min":[daily_f["ano"].min() if not daily_f.empty else None],
    "ano_max":[daily_f["ano"].max() if not daily_f.empty else None],
})
resumo.to_csv(os.path.join(TCC_DIR, "resumo_cobertura.csv"), index=False, encoding="utf-8")
display(resumo)


In [ ]:
# Célula 3 — Qualidade e correlações
import os, numpy as np, pandas as pd, matplotlib.pyplot as plt
TCC_DIR = os.path.join(OUT_DIR, "tcc")
os.makedirs(TCC_DIR, exist_ok=True)

cov = (daily.groupby("cidade")["data"].nunique().rename("dias_disponiveis")).reset_index()
cov.to_csv(os.path.join(TCC_DIR,"01_cobertura_dias_por_cidade.csv"), index=False, encoding="utf-8")

qtab = (daily.groupby("cidade")
        .agg(
            dias=("data","nunique"),
            mediana_kj=("daily_radiacao_kj","median"),
            media_kj=("daily_radiacao_kj","mean"),
            p95_kj=("daily_radiacao_kj", lambda s: np.nanpercentile(s.dropna(),95)),
        ).reset_index())
qtab.to_csv(os.path.join(TCC_DIR,"02_quality_medidas_radiacao_por_cidade.csv"), index=False, encoding="utf-8")

corr_cols = ["temp_c_mean","umid_pct_mean","vento_ms_mean","pressao_mb_mean","precip_mm_sum","daily_radiacao_kj"]
corr_city_list = []
for city, g in daily.groupby("cidade"):
    gg = g[corr_cols].dropna()
    if len(gg)>=60:
        r = gg.corr(method="pearson")["daily_radiacao_kj"].drop("daily_radiacao_kj")
        out = {"cidade":city}
        out.update({f"corr_{k}":v for k,v in r.items()})
        corr_city_list.append(out)
corr_city_df = pd.DataFrame(corr_city_list)
corr_city_df.to_csv(os.path.join(TCC_DIR,"03_correlacoes_por_cidade.csv"), index=False, encoding="utf-8")

gg = daily[corr_cols].dropna()
glob_corr = gg.corr(method="pearson")
glob_corr.to_csv(os.path.join(TCC_DIR,"04_correlacao_global.csv"), encoding="utf-8")

plt.figure(figsize=(6,5))
plt.imshow(glob_corr.values, interpolation="nearest")
plt.xticks(range(len(glob_corr.columns)), glob_corr.columns, rotation=45, ha="right")
plt.yticks(range(len(glob_corr.index)), glob_corr.index)
plt.colorbar()
plt.tight_layout()
plt.savefig(os.path.join(TCC_DIR,"04_correlacao_global_heatmap.png"), dpi=180, bbox_inches="tight"); plt.close()


In [ ]:
# Célula 4 — Clusterização k=5 (robusta com poucos/nenhum caso)
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os

city_stats_f = (
    daily_f.groupby("cidade")
         .agg(
             mediana_kj=("daily_radiacao_kj","median"),
             p75_kj=("daily_radiacao_kj", lambda s: np.nanpercentile(s.dropna(),75)),
             p95_kj=("daily_radiacao_kj", lambda s: np.nanpercentile(s.dropna(),95)),
             media_kj=("daily_radiacao_kj","mean"),
             std_kj=("daily_radiacao_kj","std"),
             iqr_kj=("daily_radiacao_kj", lambda s: np.nanpercentile(s.dropna(),75)-np.nanpercentile(s.dropna(),25)),
             temp_c_mean=("temp_c_mean","mean"),
             umid_pct_mean=("umid_pct_mean","mean"),
             vento_ms_mean=("vento_ms_mean","mean"),
             pressao_mb_mean=("pressao_mb_mean","mean"),
             precip_mm_mean=("precip_mm_sum","mean"),
             dias=("data","nunique")
         )
         .reset_index()
)

if city_stats_f.empty or city_stats_f["cidade"].nunique()==0:
    print("⚠️ Sem cidades para clusterizar. Pulando KMeans.")
    city_stats_f["cluster_rank"] = np.nan
    city_stats_f["cluster_label"] = "Indefinido"
else:
    Xc = city_stats_f.drop(columns=["cidade"]).copy()
    # remove colunas totalmente NaN
    Xc = Xc.loc[:, Xc.notna().any(axis=0)]
    # substitui NaN por mediana da coluna
    Xc = Xc.apply(lambda col: col.fillna(col.median()) if col.dtype.kind in "fc" else col)
    # remove colunas constantes (variância zero)
    var = Xc.var(numeric_only=True)
    keep_cols = var.index[var > 0].tolist()
    Xc = Xc[keep_cols] if keep_cols else Xc.iloc[:, :0]

    n_cidades = city_stats_f.shape[0]
    if Xc.shape[0] < 2 or Xc.shape[1] == 0:
        print("⚠️ Dados insuficientes para KMeans. Atribuindo rótulo 'Indefinido'.")
        city_stats_f["cluster_rank"] = np.nan
        city_stats_f["cluster_label"] = "Indefinido"
    else:
        k = min(5, n_cidades)
        scaler_c = StandardScaler()
        Xc_s = scaler_c.fit_transform(Xc)
        km = KMeans(n_clusters=k, random_state=42, n_init=20)
        city_stats_f["cluster"] = km.fit_predict(Xc_s)

        # ordenar clusters por mediana de radiação
        cl_order = city_stats_f.groupby("cluster")["mediana_kj"].median().sort_values().index.tolist()
        rank_map = {cl: i for i,cl in enumerate(cl_order)}
        city_stats_f["cluster_rank"] = city_stats_f["cluster"].map(rank_map)

        label_scale = {0:"Muito ruim",1:"Ruim",2:"Médio",3:"Bom",4:"Muito bom"}
        if k < 5:
            label_scale = {i: ["Muito ruim","Ruim","Médio","Bom","Muito bom"][i] for i in range(k)}
        city_stats_f["cluster_label"] = city_stats_f["cluster_rank"].map(label_scale)

city_stats_f = city_stats_f.sort_values(["cluster_rank","mediana_kj"], ascending=[True,False]).reset_index(drop=True)
city_stats_f.to_csv(os.path.join(TCC_DIR,"05_cluster_cidades_k5.csv"), index=False, encoding="utf-8")

if city_stats_f["cluster_label"].nunique()>1 and city_stats_f["cluster_label"].notna().any():
    plt.figure(figsize=(8,4))
    for lab in sorted(city_stats_f["cluster_label"].dropna().unique()):
        sel = city_stats_f[city_stats_f["cluster_label"]==lab]
        plt.scatter(sel["media_kj"], sel["p95_kj"], label=lab, s=40)
    plt.xlabel("Média diária (kJ/m²)"); plt.ylabel("P95 diária (kJ/m²)"); plt.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(TCC_DIR,"05_cluster_disp_media_p95.png"), dpi=180, bbox_inches="tight"); plt.close()
else:
    print("ℹ️ Sem variabilidade suficiente para o gráfico de clusters.")


In [ ]:
# Célula 5 — Defasagens (7 dias) com daily_f
import numpy as np, pandas as pd

def add_lags(df_city, cols, max_lag=7):
    out = df_city.copy()
    for c in cols:
        for k in range(1, max_lag+1):
            out[f"{c}_lag{k}"] = out[c].shift(k)
    return out

ds = []
for city, g in daily_f.groupby("cidade"):
    g = g.sort_values("data")
    g2 = add_lags(g, ["daily_radiacao_kj","temp_c_mean","umid_pct_mean","vento_ms_mean","pressao_mb_mean","precip_mm_sum"], max_lag=7)
    g2["cidade"] = city
    ds.append(g2)

lag_f = pd.concat(ds, ignore_index=True).sort_values(["cidade","data"]).reset_index(drop=True)
lag_f = lag_f.dropna().reset_index(drop=True)
lag_f.head()


In [ ]:
# Célula 6 — Walk-Forward (ano a ano) com lag_f | robusto (sem 'squared', com early_stopping)
import warnings, numpy as np, pandas as pd, matplotlib.pyplot as plt, os
from sklearn.ensemble import RandomForestRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.exceptions import ConvergenceWarning

warnings.filterwarnings("ignore", category=ConvergenceWarning)

TCC_DIR = os.path.join(OUT_DIR, "tcc"); os.makedirs(TCC_DIR, exist_ok=True)

def rmse_np(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

years = sorted(lag_f["ano"].unique())
years = [y for y in years if YEAR_START < y <= YEAR_END]

preds_all = []
metrics_year = []

for ytest in years:
    train = lag_f[lag_f["ano"] < ytest].copy()
    test = lag_f[lag_f["ano"] == ytest].copy()
    if len(train)==0 or len(test)==0:
        continue

    y_tr = train["daily_radiacao_kj"].values
    y_te = test["daily_radiacao_kj"].values
    X_tr = train.drop(columns=["cidade","data","ano","daily_radiacao_kj"])
    X_te = test.drop(columns=["cidade","data","ano","daily_radiacao_kj"])

    rf = RandomForestRegressor(n_estimators=500, random_state=42, n_jobs=-1)
    rf.fit(X_tr, y_tr)
    pred_rf = rf.predict(X_te)

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_te_s = scaler.transform(X_te)
    mlp = MLPRegressor(
        hidden_layer_sizes=(256,128),
        activation="relu",
        learning_rate_init=1e-3,
        max_iter=500,
        early_stopping=True,
        n_iter_no_change=15,
        validation_fraction=0.1,
        random_state=42,
        tol=1e-4
    )
    mlp.fit(X_tr_s, y_tr)
    pred_mlp = mlp.predict(X_te_s)

    mae_rf  = mean_absolute_error(y_te, pred_rf)
    rmse_rf = rmse_np(y_te, pred_rf)
    r2_rf   = r2_score(y_te, pred_rf)

    mae_mlp  = mean_absolute_error(y_te, pred_mlp)
    rmse_mlp = rmse_np(y_te, pred_mlp)
    r2_mlp   = r2_score(y_te, pred_mlp)

    metrics_year.append({
        "ano": ytest,
        "RF_MAE": mae_rf,  "RF_RMSE": rmse_rf,  "RF_R2": r2_rf,
        "MLP_MAE": mae_mlp,"MLP_RMSE": rmse_mlp,"MLP_R2": r2_mlp
    })

    fold_out = test[["cidade","data","daily_radiacao_kj"]].copy()
    fold_out["ano"] = ytest
    fold_out["pred_rf"] = pred_rf
    fold_out["pred_mlp"] = pred_mlp
    preds_all.append(fold_out)

if len(preds_all)==0:
    raise RuntimeError("Sem dobras válidas no walk-forward. Verifique anos disponíveis em 'lag_f'.")

preds_all = pd.concat(preds_all, ignore_index=True)
metrics_year = pd.DataFrame(metrics_year)

metrics_avg = pd.DataFrame({
    "modelo": ["RF","MLP"],
    "MAE":  [metrics_year["RF_MAE"].mean(),  metrics_year["MLP_MAE"].mean()],
    "RMSE": [metrics_year["RF_RMSE"].mean(), metrics_year["MLP_RMSE"].mean()],
    "R2":   [metrics_year["RF_R2"].mean(),   metrics_year["MLP_R2"].mean()],
})

metrics_year.to_csv(os.path.join(TCC_DIR,"wfv_metricas_por_ano.csv"), index=False, encoding="utf-8")
metrics_avg.to_csv(os.path.join(TCC_DIR,"wfv_metricas_medias.csv"), index=False, encoding="utf-8")

plt.figure(figsize=(7,4))
plt.plot(metrics_year["ano"], metrics_year["RF_RMSE"], marker="o", label="RF")
plt.plot(metrics_year["ano"], metrics_year["MLP_RMSE"], marker="o", label="MLP")
plt.legend(); plt.title("RMSE por ano — Walk-Forward"); plt.ylabel("RMSE (kJ/m²)")
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.savefig(os.path.join(TCC_DIR,"wfv_rmse_por_ano.png"), dpi=180, bbox_inches="tight"); plt.close()

best_model = "rf" if metrics_avg.loc[metrics_avg["modelo"]=="RF","RMSE"].values[0] <= metrics_avg.loc[metrics_avg["modelo"]=="MLP","RMSE"].values[0] else "mlp"
print("Melhor modelo médio (RMSE):", best_model.upper())


In [ ]:
# Célula 7 — Ranking previsto no último ano + peso de cobertura
import numpy as np, pandas as pd, matplotlib.pyplot as plt, os

last_year = preds_all["ano"].max()
pred_last = preds_all[preds_all["ano"]==last_year].copy()
use_col = "pred_rf" if best_model=="rf" else "pred_mlp"

agg_city_pred = (
    pred_last.groupby("cidade")
             .agg(
                 mediana_real=("daily_radiacao_kj","median"),
                 mediana_pred=(use_col,"median"),
                 p95_real=("daily_radiacao_kj", lambda s: np.nanpercentile(s.dropna(),95)),
                 p95_pred=(use_col, lambda s: np.nanpercentile(s.dropna(),95)),
                 dias=("data","nunique")
             )
             .reset_index()
)

centers = city_stats_f[["cluster_rank","mediana_kj"]].groupby("cluster_rank").median().sort_index()
edges = centers["mediana_kj"].rolling(2).mean().dropna().values
def map_label(v):
    if len(edges)<4: return "Médio"
    if v < edges[0]: return "Muito ruim"
    if v < edges[1]: return "Ruim"
    if v < edges[2]: return "Médio"
    if v < edges[3]: return "Bom"
    return "Muito bom"

agg_city_pred["classe_prevista"] = agg_city_pred["mediana_pred"].apply(map_label)
agg_city_pred["score"] = 0.7*agg_city_pred["mediana_pred"].rank(pct=True) + 0.3*agg_city_pred["p95_pred"].rank(pct=True)

cov_map = coverage.set_index("cidade")["pct_dias_10anos"]/100.0
agg_city_pred["peso_cov"] = agg_city_pred["cidade"].map(cov_map).fillna(0.0).clip(0,1)
agg_city_pred["score_ponderado"] = agg_city_pred["score"] * agg_city_pred["peso_cov"]

ranking_prev = agg_city_pred.sort_values(["classe_prevista","score_ponderado"], ascending=[True,False]).reset_index(drop=True)

preds_all.to_csv(os.path.join(TCC_DIR,"wfv_previsoes_todas_as_cidades.csv"), index=False, encoding="utf-8")
agg_city_pred.to_csv(os.path.join(TCC_DIR,"wfv_agregado_cidades_ultimo_ano.csv"), index=False, encoding="utf-8")
ranking_prev.to_csv(os.path.join(TCC_DIR,"wfv_ranking_previsto_ultimo_ano.csv"), index=False, encoding="utf-8")

top10 = ranking_prev.sort_values("score_ponderado", ascending=False).head(10)
plt.figure(figsize=(9,4))
plt.bar(top10["cidade"], top10["mediana_pred"])
plt.title(f"Top 10 — potencial previsto ponderado ({last_year}, modelo: {best_model.upper()})")
plt.xticks(rotation=45, ha="right"); plt.ylabel("kJ/m²")
plt.tight_layout()
plt.savefig(os.path.join(TCC_DIR,"wfv_top10_potencial_previsto_ponderado.png"), dpi=180, bbox_inches="tight"); plt.close()


In [ ]:
# Célula 8 — CNN 1D opcional (split fixo: todos < YEAR_END para prever YEAR_END)
import numpy as np, pandas as pd, tensorflow as tf
from tensorflow.keras import layers, models
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt, os

mask_train = lag["ano"] < YEAR_END
train_seq = lag[mask_train].copy()
test_seq = lag[~mask_train].copy()

y_train = train_seq["daily_radiacao_kj"].values
y_test = test_seq["daily_radiacao_kj"].values

X_train = train_seq.drop(columns=["cidade","data","ano","daily_radiacao_kj"])
X_test = test_seq.drop(columns=["cidade","data","ano","daily_radiacao_kj"])

lag_cols = [c for c in X_train.columns if c.endswith(tuple([f"_lag{k}" for k in range(1,8)]))]
bases = sorted(set([c.rsplit("_lag",1)[0] for c in lag_cols]))

def to_seq(df_):
    arrs = []
    for base in bases:
        mats = [df_[f"{base}_lag{k}"].values.reshape(-1,1) for k in range(7,0,-1)]
        arrs.append(np.concatenate(mats, axis=1)[:, :, None])
    Xseq = np.concatenate(arrs, axis=2)
    return Xseq

Xtr_seq = to_seq(X_train)
Xte_seq = to_seq(X_test)

cnn = models.Sequential([
    layers.Input(shape=(7, Xtr_seq.shape[2])),
    layers.Conv1D(64, 3, padding="causal", activation="relu"),
    layers.Conv1D(64, 3, padding="causal", activation="relu"),
    layers.GlobalAveragePooling1D(),
    layers.Dense(128, activation="relu"),
    layers.Dense(1)
])
cnn.compile(optimizer=tf.keras.optimizers.Adam(1e-3), loss="mae")
hist = cnn.fit(Xtr_seq, y_train, epochs=20, batch_size=256, verbose=0, validation_split=0.1)
pred_cnn = cnn.predict(Xte_seq, verbose=0).ravel()

mae_cnn = mean_absolute_error(y_test, pred_cnn)
rmse_cnn = mean_squared_error(y_test, pred_cnn, squared=False)
r2_cnn = r2_score(y_test, pred_cnn)

pd.DataFrame({"modelo":["CNN1D"],"MAE":[mae_cnn],"RMSE":[rmse_cnn],"R2":[r2_cnn]}).to_csv(
    os.path.join(TCC_DIR,"cnn_metricas_split_fixo.csv"), index=False, encoding="utf-8"
)

plt.figure(figsize=(6,4))
plt.plot(hist.history["loss"], label="treino")
plt.plot(hist.history["val_loss"], label="val")
plt.title("CNN1D — curva de aprendizado (MAE)")
plt.legend()
plt.tight_layout()
plt.savefig(os.path.join(TCC_DIR,"cnn_curva_aprendizado.png"), dpi=180, bbox_inches="tight"); plt.close()


In [ ]:
# Célula 9 — Insolação por cidade (média, mediana, min, max) usando daily_f
import os, pandas as pd, numpy as np, matplotlib.pyplot as plt
TCC_DIR = os.path.join(OUT_DIR, "tcc"); os.makedirs(TCC_DIR, exist_ok=True)

stats_city = (daily_f.groupby("cidade")["daily_radiacao_kj"]
                   .agg(media_kj="mean", mediana_kj="median", minimo_kj="min", maximo_kj="max", dias="count")
                   .reset_index()).sort_values("media_kj", ascending=False).reset_index(drop=True)
stats_city.to_csv(os.path.join(TCC_DIR,"insolacao_estatisticas_por_cidade.csv"), index=False, encoding="utf-8")

topN = 10
top_media = stats_city.head(topN)
bot_media = stats_city.tail(topN).iloc[::-1]

plt.figure(figsize=(9,4))
plt.bar(top_media["cidade"], top_media["media_kj"])
plt.title("Top 10 — média diária de radiação (kJ/m²)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(TCC_DIR,"insolacao_top10_media.png"), dpi=180, bbox_inches="tight"); plt.close()

plt.figure(figsize=(9,4))
plt.bar(bot_media["cidade"], bot_media["media_kj"])
plt.title("Bottom 10 — média diária de radiação (kJ/m²)")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.savefig(os.path.join(TCC_DIR,"insolacao_bottom10_media.png"), dpi=180, bbox_inches="tight"); plt.close()


In [ ]:
# Célula 10 — Inventário de arquivos para anexar no TCC
saved = sorted([os.path.join(TCC_DIR,f) for f in os.listdir(TCC_DIR)])
pd.DataFrame({"arquivos_salvos": saved})
